# Notebook 01 — Cos'è l'IA generativa e gli LLM

**Modulo:** 01
**Tempo stimato:** 1-1,5 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Un `.env` funzionante con le chiavi dei tre provider principali
- La tua prima chiamata reale a un LLM (Claude, di Anthropic)
- Una seconda chiamata allo stesso prompt su un provider diverso (GPT, di OpenAI)
- La capacità di leggere una risposta API e riconoscere a occhio nudo i concetti
  del Modulo 01: prompt, modello, inferenza, completion, context window

## Prerequisiti

```bash
pip install anthropic openai python-dotenv
```

Crea un file `.env` nella cartella del progetto con questo contenuto
(sostituisci con le tue chiavi reali):

```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
GOOGLE_API_KEY=...
```

Useremo `GOOGLE_API_KEY` più avanti nel corso (a partire dal Modulo 04):
qui la verifichiamo solo per abituarci fin da subito a tenere pronte tutte e
tre le chiavi nello stesso `.env`.


---
## Cella 0 — Setup: caricamento e verifica delle chiavi

Esegui sempre questa cella per prima. Se una chiave manca, il notebook te lo
segnala subito invece di farti scoprire l'errore tre celle più avanti.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")
GOOGLE_KEY = os.getenv("GOOGLE_API_KEY")

def verifica_chiave(nome: str, chiave: str | None) -> None:
    """Mostra solo i primi caratteri della chiave: mai stampare una chiave per intero."""
    if chiave:
        print(f"OK  {nome}: {chiave[:8]}...")
    else:
        print(f"MANCA  {nome}: non trovata. Controlla il tuo file .env")

verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)
verifica_chiave("ANTHROPIC_API_KEY", ANTHROPIC_KEY)
verifica_chiave("GOOGLE_API_KEY", GOOGLE_KEY)


---
## Cella 1 — La tua prima chiamata a un LLM

Chiediamo a Claude (Anthropic) di rispondere a una domanda semplice.
Osserva con attenzione i quattro concetti del Modulo 01 mentre leggi il codice:

- il **prompt** è il testo dentro `messages`
- il **modello** è il valore del parametro `model`
- l'**inferenza** è ciò che succede sui server di Anthropic tra l'invio della
  richiesta e la ricezione della risposta (non lo vedi, ma è lì che il
  modello genera il testo token per token)
- la **completion** è `risposta.content[0].text`: il risultato finale

**Prima di eseguire:** cosa ti aspetti che risponda? Scrivilo mentalmente,
poi confronta con l'output reale.

In [ ]:
import anthropic

cliente_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

prompt = "Spiega in due frasi cos'è un token per un LLM, come se lo spiegassi a un principiante."

risposta = cliente_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",  # verifica l'ID modello aggiornato nella doc ufficiale
    max_tokens=200,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

completion = risposta.content[0].text

print("PROMPT:")
print(prompt)
print()
print("COMPLETION:")
print(completion)
print()
print("Token di input (il prompt tokenizzato):", risposta.usage.input_tokens)
print("Token di output (la completion tokenizzata):", risposta.usage.output_tokens)


**Cosa osservare:** `usage.input_tokens` e `usage.output_tokens` non sono
il numero di parole del prompt e della risposta: sono il numero di *token*,
l'unità che il modello elabora davvero. È la stessa idea di context window
vista nella dispensa: la somma di questi due numeri deve stare dentro il
limite massimo che il modello può gestire in una singola interazione.

---
## Cella 2 — Lo stesso prompt, un provider diverso

Poniamo la stessa identica domanda a GPT (OpenAI). Il concetto di
prompt → inferenza → completion è identico: cambia solo il fornitore, il
nome del modello e i dettagli sintattici della risposta (approfondiremo
queste differenze nel Modulo 04 e, in modo sistematico, nel Modulo 08).

In [ ]:
from openai import OpenAI

cliente_openai = OpenAI(api_key=OPENAI_KEY)

risposta_gpt = cliente_openai.chat.completions.create(
    model="gpt-5-mini",  # verifica l'ID modello aggiornato nella doc ufficiale
    max_completion_tokens=200,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

completion_gpt = risposta_gpt.choices[0].message.content

print("PROMPT (identico al precedente):")
print(prompt)
print()
print("COMPLETION (GPT):")
print(completion_gpt)
print()
print("Token di input:", risposta_gpt.usage.prompt_tokens)
print("Token di output:", risposta_gpt.usage.completion_tokens)


**Domanda per te:** le due risposte (Claude e GPT) dicono la stessa cosa
con parole diverse, o si concentrano su aspetti diversi del concetto di
token? Non c'è una risposta "giusta" da cercare qui: è il primo assaggio di
un'osservazione che approfondiremo con calma nel Modulo 04.

---
## Cella 3 — Esercizio: cambia il prompt, osserva la context window

**TODO:** modifica la variabile `prompt_lungo` qui sotto scrivendo un testo
di almeno 200-300 parole (per esempio, incolla un paragrafo da un articolo
che hai a portata di mano, o fanne generare uno a un LLM che già usi). Poi
esegui la cella e osserva quanti token vengono conteggiati in input.

L'obiettivo è collegare con i tuoi occhi l'idea "più testo scrivo, più
token consumo in input" — che è esattamente il motivo per cui la context
window ha un limite.

In [ ]:
# TODO: sostituisci questo testo con un paragrafo di almeno 200-300 parole
prompt_lungo = """
Scrivi qui il tuo testo lungo...
"""

risposta_lunga = cliente_anthropic.messages.create(
    model="claude-haiku-4-5-20251001",  # verifica l'ID modello aggiornato nella doc ufficiale
    max_tokens=100,
    messages=[
        {"role": "user", "content": f"Riassumi questo testo in una frase: {prompt_lungo}"}
    ]
)

print("Token di input consumati dal prompt lungo:", risposta_lunga.usage.input_tokens)
print("Completion:", risposta_lunga.content[0].text)


---
## Sfida finale — Esplora inferenza e completion con tre domande

Costruisci un piccolo script che:

1. Definisce una lista di 3 prompt diversi (a tua scelta, per esempio una
   domanda fattuale, una richiesta creativa, un piccolo calcolo).
2. Per ciascun prompt, chiama Claude e stampa: il prompt, la completion, e i
   token di input/output.
3. Alla fine, stampa il totale di token di input e output usati in tutto
   lo script.

Questo è lo stesso schema delle Celle 1-2, ripetuto in un ciclo: è
l'occasione per abituarti a leggere `usage` come un'informazione normale,
non come un dettaglio a parte.

In [ ]:
domande = [
    "Qual è la capitale dell'Australia?",
    "Inventa un nome per una startup che aiuta le persone a leggere di più.",
    "Se ho 3 mele e ne mangio una, quante me ne restano? Rispondi solo col numero.",
]

totale_input = 0
totale_output = 0

for domanda in domande:
    r = cliente_anthropic.messages.create(
        model="claude-haiku-4-5-20251001",  # verifica l'ID modello aggiornato nella doc ufficiale
        max_tokens=100,
        messages=[{"role": "user", "content": domanda}]
    )
    testo = r.content[0].text
    print(f"Prompt: {domanda}")
    print(f"Completion: {testo}")
    print(f"Token: {r.usage.input_tokens} input / {r.usage.output_tokens} output")
    print("-" * 50)

    totale_input += r.usage.input_tokens
    totale_output += r.usage.output_tokens

print(f"\nTotale token di input usati:  {totale_input}")
print(f"Totale token di output usati: {totale_output}")


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 01 — Modulo 01 del Corso Completo di IA Generativa*